In [1]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma2:2b",
        "prompt": "Say hello in one sentence.",
        "stream": False
    }
)

print(response.json()["response"])

Hello! 🌻 



In [2]:
snippets = [
    {
        "language": "python",
        "code": """
def divide(a, b):
    return a / b

result = divide(10, 0)
print(result)
"""
    },
    {
        "language": "python",
        "code": """
def get_user(users, id):
    for u in users:
        if u['id'] == id:
            return u
"""
    },
    {
        "language": "javascript",
        "code": """
function calculateTotal(items) {
    var total = 0;
    for (var i = 0; i < items.length; i++) {
        total = total + items[i].price;
    }
    return total;
}
"""
    },
    {
        "language": "javascript",
        "code": """
function fetchData(url) {
    fetch(url).then(res => res.json()).then(data => {
        console.log(data)
    })
}
"""
    },
    {
        "language": "cpp",
        "code": """
#include <iostream>
using namespace std;

int main() {
    int arr[5];
    for (int i = 0; i <= 5; i++) {
        arr[i] = i * 2;
        cout << arr[i] << endl;
    }
    return 0;
}
"""
    },
    {
        "language": "python",
        "code": """
def process_items(items):
    result = []
    for i in range(len(items)):
        for j in range(len(items)):
            if items[i] == items[j] and i != j:
                result.append(items[i])
    return result
"""
    },
]

print(f"Total snippets: {len(snippets)}")

Total snippets: 6


In [3]:
def build_prompt(code, language):
    return f"""You are a senior code reviewer. Analyze this {language} code and identify the main problem.

Code:
{code}

Respond in this exact format:
WHAT_IS_WRONG: <one sentence>
WHY_IT_IS_WRONG: <one sentence explaining the underlying issue>
"""

In [4]:
results = []

for idx, item in enumerate(snippets):
    prompt = build_prompt(item["code"], item["language"])
    
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma2:2b",
            "prompt": prompt,
            "stream": False
        }
    )
    
    output_text = response.json()["response"]
    
    results.append({
        "index": idx,
        "language": item["language"],
        "output": output_text
    })
    
    print(f"--- Snippet {idx} ({item['language']}) ---")
    print(output_text)
    print()

--- Snippet 0 (python) ---
WHAT_IS_WRONG: The code will cause a ZeroDivisionError. 
WHY_IT_IS_WRONG: Attempting to divide by zero is mathematically impossible and will lead to an error in Python.  


--- Snippet 1 (python) ---
WHAT_IS_WRONG: The function might return unexpected results due to potential data discrepancies and lack of error handling. 

WHY_IT_IS_WRONG: The code relies on a simple loop, searching for a specific `id` within a list of `users`, but it doesn't include any checks for the existence or validity of the  `users` input, nor does it handle potential `IndexError` if the `id` is not present in the `users`.


--- Snippet 2 (javascript) ---
WHAT_IS_WRONG: The function uses var for its loop counter, which can lead to variable scope issues. 
WHY_IT_IS_WRONG:  Using `var` declares a global variable that might cause problems with variable scoping, leading to unexpected behavior and potential bugs. It's recommended to use `let` or `const` for better code maintainability and 

In [5]:
TAXONOMY = [
    "missing_error_handling",   # try/except, .catch(), null checks missing
    "no_input_validation",       # function doesn't validate inputs before use
    "outdated_syntax",           # var instead of let/const, deprecated patterns
    "array_bounds_error",        # off-by-one, out-of-bounds access
    "inefficient_loop",          # O(n²) or worse when better exists
]

print("Taxonomy finalized:", TAXONOMY)

Taxonomy finalized: ['missing_error_handling', 'no_input_validation', 'outdated_syntax', 'array_bounds_error', 'inefficient_loop']


In [6]:
def build_json_prompt(code, language):
    return f"""You are a senior code reviewer. Analyze the following {language} code and respond ONLY with a valid JSON object — no extra text, no markdown, no explanation outside the JSON.

The JSON must have exactly these 4 keys:
- "what_is_wrong": one sentence describing the specific issue
- "why_it_is_wrong": one sentence explaining the underlying reason/risk
- "how_to_fix": one sentence with a concrete fix
- "concept_to_study": a short topic name (2-4 words) the developer should study

Code:
{code}

Respond with ONLY the JSON object, nothing else."""

In [7]:
def get_json_feedback(code, language):
    prompt = build_json_prompt(code, language)
    
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma2:2b",
            "prompt": prompt,
            "format": "json",
            "stream": False
        }
    )
    
    return response.json()["response"]

# Test on one snippet first
test_output = get_json_feedback(snippets[0]["code"], snippets[0]["language"])
print(test_output)

{"what_is_wrong": "The code attempts to divide by zero, which will raise a ZeroDivisionError.", "why_it_s_wrong": "Dividing by zero is undefined in mathematics and will cause unpredictable program behavior.", "how_to_fix": "Add a check for zero before performing the division: if b == 0 then return 0 or handle it appropriately (e.g., raise an error).", "concept_to_study": "Error Handling"}


In [8]:
import json

json_results = []
failures = 0

for idx, item in enumerate(snippets):
    raw_output = get_json_feedback(item["code"], item["language"])
    
    try:
        parsed = json.loads(raw_output)
        json_results.append({"index": idx, "success": True, "data": parsed})
    except json.JSONDecodeError:
        failures += 1
        json_results.append({"index": idx, "success": False, "raw": raw_output})
    
    print(f"Snippet {idx}: {'OK' if json_results[-1]['success'] else 'FAILED TO PARSE'}")

print(f"\nTotal failures: {failures}/{len(snippets)}")

Snippet 0: OK
Snippet 1: OK
Snippet 2: OK
Snippet 3: OK
Snippet 4: OK
Snippet 5: OK

Total failures: 0/6


In [9]:
from pydantic import BaseModel, ValidationError

class FeedbackSchema(BaseModel):
    what_is_wrong: str
    why_it_is_wrong: str
    how_to_fix: str
    concept_to_study: str

def validate_feedback(raw_output: str):
    try:
        data = json.loads(raw_output)
        validated = FeedbackSchema(**data)
        return {"success": True, "data": validated.dict()}
    except (json.JSONDecodeError, ValidationError) as e:
        return {"success": False, "error": str(e)}

# Test on the snippet that had the field-name issue
result = validate_feedback(test_output)
print(result)

{'success': False, 'error': "1 validation error for FeedbackSchema\nwhy_it_is_wrong\n  Field required [type=missing, input_value={'what_is_wrong': 'The co...tudy': 'Error Handling'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing"}


In [10]:
def get_validated_feedback(code, language, max_retries=1):
    for attempt in range(max_retries + 1):
        raw_output = get_json_feedback(code, language)
        result = validate_feedback(raw_output)
        if result["success"]:
            return result
    # Agar sab attempts fail ho jayein
    return {"success": False, "status": "parsing_failed", "last_raw": raw_output}

# Test again
final_result = get_validated_feedback(snippets[0]["code"], snippets[0]["language"])
print(final_result)

{'success': False, 'status': 'parsing_failed', 'last_raw': '{"what_is_wrong": "The code will raise a ZeroDivisionError when dividing by zero.", "why_it_s_wrong": "Attempting division by zero is mathematically undefined and can lead to unexpected program crashes.", "how_to_fix": "Add a check for zero before performing the division, preventing potential errors.", "concept_to_study": "Exception Handling"}'}
